### Search Engine With Tools And Agents

In [1]:
## Arxiv--Research
## Tools creation
from langchain_classic.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_classic.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [2]:
import wikipedia
wikipedia.set_user_agent("rtool/0.1 (arpansingh6944@gmail.com)")


In [3]:
## Used the inbuilt tool of wikipedia
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki,handle_tool_error=True)
wiki.name

'wikipedia'

In [4]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [5]:
tools=[wiki,arxiv]

In [6]:
## Custom tools[RAG Tool]
from langchain_classic.document_loaders import WebBaseLoader
from langchain_classic.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

c:\Users\win11\OneDrive\Desktop\rtool\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1631.43it/s]


In [8]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents,embeddings)
retriever=vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002F1F41E9E80>, search_kwargs={})

In [9]:
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith-search","Search any information about Langsmith ")

retriever_tool.name

'langsmith-search'

In [10]:
tools=[wiki,arxiv,retriever_tool]


In [11]:
tools

[WikipediaQueryRun(handle_tool_error=True, api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\win11\\OneDrive\\Desktop\\rtool\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about Langsmith ', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000002F1F73E4880>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000002F1F88BB7F0>)]

In [12]:
## Run all this tools with Agents and LLM Models

## Tools, LLM-->AgentExecutor
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import openai
load_dotenv()
import os

groq_api_key=os.getenv("GROQ_API_KEY")
openai.api_key=os.getenv("OPENAI_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b")

In [13]:
## Prompt Template
'''from langchain_classic import hub
prompt=hub.pull("hwchase17/openai-functions-agent",dangerously_pull_public_prompt=True)
prompt.messages'''

'from langchain_classic import hub\nprompt=hub.pull("hwchase17/openai-functions-agent",dangerously_pull_public_prompt=True)\nprompt.messages'

In [14]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt("hwchase17/openai-functions-agent",dangerously_pull_public_prompt=True)
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [15]:
## Agents
from langchain_classic.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm,tools,prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')] | typing.Annotated[langchain_core.messages.chat

In [16]:
## Agent Executer
from langchain_classic.agents import AgentExecutor
agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True,handle_parsing_errors=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='

In [17]:
agent_executor.invoke({"input":"Tell me about Langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'Langsmith'}`


LangSmith Observability - Docs by LangChain

root cause, and resolve them with LangSmith Engine.If your applications call models through the LLM Gateway, the gateway traces every call itself, with no tracing setup in your code. For where those traces appear and what they record, see View LLM Gateway traces.For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing.To set up a LangSmith instance, visit the Platform setup section to choose between cloud, hybrid, or self-hosted. All options include observability, evaluation, prompt engineering, and deployment.

Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chanc

{'input': 'Tell me about Langsmith',
 'output': '**LangSmith – the LangChain observability platform**\n\n| What it is | Why you’d use it |\n|------------|------------------|\n| **LangSmith** is an end‑to‑end observability and evaluation platform built by the LangChain team.  It lives in the same ecosystem as LangChain, but it’s a *stand‑alone service* that can be used with any LLM‑powered app. | It gives you a single place to see, debug, and improve the behaviour of your LLMs in production.  Think of it as the “telemetry dashboard” for generative‑AI services. |\n\n---\n\n## Core capabilities\n\n| Feature | Description |\n|---------|-------------|\n| **Tracing** | Every call to an LLM (or a chain of calls) is automatically captured as a *trace*.  Traces show the full request/response flow, timestamps, token counts, costs, and any custom metadata you attach. |\n| **Debugging & Monitoring** | Inspect failed or sub‑optimal runs, compare performance over time, and set alerts on key metrics 

In [18]:
agent_executor.invoke({"input":"What is machine learning"})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'machine learning definition'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks wi**Machine learning** is a subfield of artificial intelligence (AI) that focuses on designing algorithms and statistical models that enable computers to improve their performance on a specific task through experience—i.e., by learning from data rather than being explicitly programmed for every possible scenario.

Key points:

| Aspect | What it means |
|--------|---------------|
| **Data‑driven** | Models are built from large datasets, capturing patterns or relationships that would be too complex or time‑consuming to encode manually. |
| **Statistical learning** | Uses probability, statistics, and optimization to infer rules that 

{'input': 'What is machine learning',
 'output': '**Machine learning** is a subfield of artificial intelligence (AI) that focuses on designing algorithms and statistical models that enable computers to improve their performance on a specific task through experience—i.e., by learning from data rather than being explicitly programmed for every possible scenario.\n\nKey points:\n\n| Aspect | What it means |\n|--------|---------------|\n| **Data‑driven** | Models are built from large datasets, capturing patterns or relationships that would be too complex or time‑consuming to encode manually. |\n| **Statistical learning** | Uses probability, statistics, and optimization to infer rules that generalize to new, unseen data. |\n| **Adaptability** | Once trained, a model can adapt its predictions or decisions as more data becomes available or as the underlying distribution shifts. |\n| **Supervised, unsupervised, reinforcement** |  - *Supervised*: learns from labeled examples (e.g., classifying 

In [19]:
agent_executor.invoke({"input":"What's the paper 1706.03762 about?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c**Paper ID:** 1706.03762  
**Title:** *Attention Is All You Need*  
**Authors:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin  
**Published:** 2017 (arXiv:1706.03762)

### Core Idea
The paper introduces the **Transformer** architecture, a novel neural network model that relies entirely on **self‑attention mechanisms** and **feed‑forward layers**, dispensing with the recurrent (RNN) and convolutional (CNN) components that dominated sequence‑to‑sequence (seq2seq) tasks like machine translation.

### Key Contributions
| Contribution | What it does |
|--------------|-------------

{'input': "What's the paper 1706.03762 about?",
 'output': '**Paper ID:** 1706.03762  \n**Title:** *Attention Is All You Need*  \n**Authors:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin  \n**Published:** 2017 (arXiv:1706.03762)\n\n### Core Idea\nThe paper introduces the **Transformer** architecture, a novel neural network model that relies entirely on **self‑attention mechanisms** and **feed‑forward layers**, dispensing with the recurrent (RNN) and convolutional (CNN) components that dominated sequence‑to‑sequence (seq2seq) tasks like machine translation.\n\n### Key Contributions\n| Contribution | What it does |\n|--------------|--------------|\n| **Self‑attention (scaled dot‑product)** | Allows each token to attend to every other token in a sequence, capturing long‑range dependencies in a single layer. |\n| **Multi‑head attention** | Learns multiple, diverse attention patterns in parallel, improving represen